# Blender on Colab — headless, GPU‑ready

Blender runs fine on a free Colab VM. What it does **not** need here is a display: this notebook uses
**`bpy`**, Blender itself shipped as a Python module, so a render is just a Python call.

| | |
|---|---|
| **Runtime** | `Runtime → Change runtime type → T4 GPU` (optional — CPU works, just ~10× slower) |
| **Setup time** | ~2 min once per session (`/content` is wiped when the VM recycles) |
| **Output** | PNG/EXR frames in `/content/blender_out`, or straight to Drive |
| **Engine** | Cycles only. EEVEE needs a real GL context and will not render headless. |

**Why there is an apt step:** the `bpy` wheel links the X11/GL stack even in background mode. Without those
shared objects `import bpy` dies with `libXrender.so.1: cannot open shared object file`. Colab has root, so
this is a one‑liner.

**What breaks:** the Blender *GUI*, the add‑on panel, EEVEE, and anything that expects a window. If you need
those, see `README.md` in this folder for the Xvfb route and its trade‑offs.

Run the cells top to bottom. Each one checks its own result, so a failure stops at the cell that caused it
instead of surfacing later as a confusing render error.

In [ ]:
import subprocess, sys, os, shutil, time, glob

WORK = "/content/blender_out"
os.makedirs(WORK, exist_ok=True)

print("python :", sys.version.split()[0])

try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20)
    GPU_LINES = [l for l in gpu.stdout.strip().splitlines() if l]
except Exception:
    GPU_LINES = []

GPU_OK = bool(GPU_LINES)
print("gpu    :", GPU_LINES[0] if GPU_OK else "none (CPU runtime - set Runtime > Change runtime type to add a GPU)")

In [ ]:
# Blender's shared-library dependencies. Cheap, and idempotent across a session restart.
APT = ("libxrender1 libxi6 libxfixes3 libxxf86vm1 libsm6 libice6 libxt6 "
       "libgl1 libegl1 libxkbcommon0 libgomp1")
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y {APT} 2>&1 | tail -2
print("system libs: ok")

In [ ]:
# Install Blender. Two routes, picked automatically:
#   1. pip  "bpy"  - Blender as a Python module, same process as this notebook. Fast, needs a matching ABI.
#   2. tarball     - the official binary. Always works, but renders go through a subprocess.
BPY_PIN = "5.0.1"
# 4.2 is the LTS line (4.2.19 was current at time of writing); the list is tried in order
BLENDER_URLS = [f"https://download.blender.org/release/Blender4.2/blender-4.2.{n}-linux-x64.tar.xz"
                for n in (19, 18, 16, 12, 3)]

HAVE_BPY, BLENDER = False, None

if sys.version_info[:2] == (3, 11):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"bpy=={BPY_PIN}"],
                       capture_output=True, text=True)
    if r.returncode:
        print("pip install bpy failed:\n", r.stderr[-600:])
else:
    print(f"python {sys.version_info.major}.{sys.version_info.minor} has no bpy wheel - using the tarball")

try:
    import bpy
    HAVE_BPY = True
    print("bpy", bpy.app.version_string, "- in-process")
except Exception as e:
    print("bpy import failed:", e)

if not HAVE_BPY:
    BLENDER = shutil.which("blender")
    if not BLENDER:
        for url in BLENDER_URLS:
            probe = subprocess.run(["wget", "-q", "--spider", url], capture_output=True)
            if probe.returncode == 0:
                print("fetching", url, "(~350 MB, 1-2 min)")
                subprocess.run(["wget", "-q", "-O", "/tmp/bl.tar.xz", url], check=True)
                break
        else:
            raise RuntimeError("no Blender tarball reachable - check the URL list")
        print("extracting...")
        subprocess.run(["tar", "-xf", "/tmp/bl.tar.xz", "-C", "/tmp"], check=True)
        BLENDER = sorted(glob.glob("/tmp/blender-*-linux-x64/blender"))[-1]
    print("blender cli:", BLENDER)
    v = subprocess.run([BLENDER, "--version"], capture_output=True, text=True)
    print(v.stdout.splitlines()[0])

assert HAVE_BPY or BLENDER, "no Blender available"

## The scene

This is plain Blender Python — the same script would run inside the Blender GUI, or via
`blender --background --python scene.py`. It is a template so the next cell can re-render it at any
resolution / sample count / device without edits.

It shows the four things every headless setup gets wrong: a **camera aimed by quaternion** (not by
eyeballing euler angles), a **metallic material that needs something to reflect**, **area lights with real
sizes** (a bare sun makes everything look plastic), and **`view_transform = "AgX"`** so highlights roll off
instead of clipping to white.

In [ ]:
SCENE = r"""
import bpy, math
from mathutils import Vector

OUT     = "__OUT__"
RES     = __RES__
SAMPLES = __SAMPLES__
DEVICE  = "__DEVICE__"

bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene

scene.render.engine = "CYCLES"
scene.cycles.samples = SAMPLES
scene.cycles.use_denoising = True
scene.render.resolution_x, scene.render.resolution_y = RES
scene.render.resolution_percentage = 100
scene.render.image_settings.file_format = "PNG"
scene.render.image_settings.color_mode = "RGB"
scene.render.filepath = OUT
scene.view_settings.view_transform = "AgX"

if DEVICE == "GPU":
    prefs = bpy.context.preferences.addons["cycles"].preferences
    avail = [t[0] for t in prefs.get_device_types(bpy.context)]
    prefs.compute_device_type = "OPTIX" if "OPTIX" in avail else "CUDA"
    prefs.get_devices()
    for d in prefs.devices:
        d.use = True
    scene.cycles.device = "GPU"
    print("cycles device: GPU", prefs.compute_device_type,
          [d.name for d in prefs.devices if d.use])
else:
    scene.cycles.device = "CPU"
    print("cycles device: CPU")

# ---------- world ----------
world = bpy.data.worlds.new("World")
scene.world = world
world.use_nodes = True
bg = world.node_tree.nodes["Background"]
bg.inputs[0].default_value = (0.015, 0.017, 0.022, 1.0)
bg.inputs[1].default_value = 1.0

# ---------- floor ----------
bpy.ops.mesh.primitive_plane_add(size=40, location=(0, 0, 0))
floor = bpy.context.object
floor.name = "floor"
m = bpy.data.materials.new("floor_mat")
m.use_nodes = True
bsdf = m.node_tree.nodes["Principled BSDF"]
bsdf.inputs["Base Color"].default_value = (0.045, 0.047, 0.055, 1.0)
bsdf.inputs["Roughness"].default_value = 0.30
floor.data.materials.append(m)

# ---------- brushed metal sphere ----------
bpy.ops.mesh.primitive_uv_sphere_add(radius=1.0, segments=96, ring_count=48,
                                     location=(0, 0, 1.0))
sphere = bpy.context.object
sphere.name = "metal_sphere"
bpy.ops.object.shade_smooth()
m = bpy.data.materials.new("metal_mat")
m.use_nodes = True
bsdf = m.node_tree.nodes["Principled BSDF"]
bsdf.inputs["Base Color"].default_value = (0.92, 0.90, 0.88, 1.0)
bsdf.inputs["Metallic"].default_value = 1.0
bsdf.inputs["Roughness"].default_value = 0.12
sphere.data.materials.append(m)

# ---------- matte block, for colour separation ----------
bpy.ops.mesh.primitive_cube_add(size=1.6, location=(-2.4, 0.9, 0.8))
block = bpy.context.object
block.name = "block"
block.rotation_euler = (0, 0, math.radians(24))
m = bpy.data.materials.new("block_mat")
m.use_nodes = True
bsdf = m.node_tree.nodes["Principled BSDF"]
bsdf.inputs["Base Color"].default_value = (0.42, 0.045, 0.035, 1.0)
bsdf.inputs["Roughness"].default_value = 0.55
block.data.materials.append(m)

# ---------- key + rim ----------
def area(name, loc, energy, size, color=(1, 1, 1)):
    bpy.ops.object.light_add(type="AREA", location=loc)
    ob = bpy.context.object
    ob.name = name
    ob.data.energy = energy
    ob.data.size = size
    ob.data.color = color
    d = Vector((0, 0, 1.0)) - ob.location
    ob.rotation_euler = d.to_track_quat("-Z", "Y").to_euler()
    return ob

area("key",  (3.6, -4.2, 5.0), 900, 5.0)
area("rim",  (-4.2, 3.1, 3.4), 700, 2.0, (0.72, 0.84, 1.0))
area("fill", (-3.2, -3.0, 1.6), 120, 3.0)

# ---------- camera ----------
bpy.ops.object.camera_add(location=(5.1, -6.3, 3.3))
cam = bpy.context.object
cam.data.lens = 70
target = Vector((0.0, 0.0, 1.0))
cam.rotation_euler = (target - cam.location).to_track_quat("-Z", "Y").to_euler()
scene.camera = cam

bpy.ops.render.render(write_still=True)
print("wrote", OUT)
"""

In [ ]:
def run_script(script, timeout=2400):
    """Execute a Blender script either in-process (bpy) or via the blender CLI."""
    if HAVE_BPY:
        exec(compile(script, "<scene>", "exec"), {"__name__": "__main__", "bpy": bpy})
    else:
        path = os.path.join(WORK, "_scene.py")
        with open(path, "w") as f:
            f.write(script)
        r = subprocess.run([BLENDER, "--background", "--factory-startup", "--python", path],
                           capture_output=True, text=True, timeout=timeout)
        keep = [l for l in r.stdout.splitlines() if ("Saved" in l or "Error" in l or "device" in l)]
        print("\n".join(keep[-6:]) or r.stderr[-900:])
        if r.returncode:
            raise RuntimeError(f"blender exited {r.returncode}")


def render(name="smoke.png", res=(480, 480), samples=48, device="AUTO", show=True):
    """Render the template scene above and return the file path."""
    if device == "AUTO":
        device = "GPU" if GPU_OK else "CPU"
    out = os.path.join(WORK, name)
    script = (SCENE.replace("__OUT__", out)
                   .replace("__RES__", repr(tuple(res)))
                   .replace("__SAMPLES__", str(samples))
                   .replace("__DEVICE__", device))
    t0 = time.time()
    run_script(script)
    dt = time.time() - t0
    size = os.path.getsize(out) / 1024 if os.path.exists(out) else 0
    print(f"{name}  {res[0]}x{res[1]}  {samples}spp  {device}  {dt:.1f}s  {size:.0f} KB  -> {out}")
    if show and size:
        from IPython.display import Image, display
        display(Image(filename=out, width=min(res[0], 640)))
    return out


smoke = render("smoke_480.png", res=(480, 480), samples=48)

If a picture of a chrome ball on a dark floor appeared above, the whole stack works: Blender, Cycles,
materials, lighting, camera, and image output.

## CPU vs GPU

Colab's free GPU is a **T4**; Cycles supports it through CUDA/OptiX. Two things to know:
the first GPU render pays a large one-time kernel-compilation cost (30–90 s), and OptiX needs
`libnvoptix` which is present on the GPU runtime but not the CPU one. The cell below renders the same
frame on both and reports the real ratio rather than a claimed one.

In [ ]:
bench = {}
for dev in (["CPU", "GPU"] if GPU_OK else ["CPU"]):
    out = os.path.join(WORK, f"bench_{dev.lower()}.png")
    script = (SCENE.replace("__OUT__", out)
                   .replace("__RES__", repr((640, 640)))
                   .replace("__SAMPLES__", "64")
                   .replace("__DEVICE__", dev))
    t0 = time.time()
    run_script(script)
    bench[dev] = time.time() - t0
    print(f"{dev}: {bench[dev]:.1f}s")

if len(bench) == 2:
    print(f"\nGPU is {bench['CPU'] / bench['GPU']:.1f}x faster on this box "
          f"(640x640, 64 spp). Note: GPU timing includes kernel compilation.")

## Rendering a `.blend` you already have

Most real use is "I have a scene, render it". Mount Drive, point at the file, and render frames. The
`bpy.ops.wm.open_mainfile` call replaces the whole scene, so nothing from the template above leaks in.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RENDERS = "/content/drive/MyDrive/blender_renders"
os.makedirs(RENDERS, exist_ok=True)
print("renders go to:", RENDERS)

In [ ]:
BLEND   = "/content/drive/MyDrive/scene.blend"   # <- your file
SAMPLES = 128
SCALE   = 100          # 100 = full resolution, 50 = half

if not os.path.exists(BLEND):
    print("no .blend at", BLEND, "- upload one to Drive, or skip this cell")
else:
    script = f"""
import bpy
bpy.ops.wm.open_mainfile(filepath={BLEND!r})
sc = bpy.context.scene
sc.render.engine = "CYCLES"
sc.cycles.samples = {SAMPLES}
sc.cycles.use_denoising = True
sc.render.resolution_percentage = {SCALE}
sc.render.image_settings.file_format = "PNG"
sc.render.filepath = {RENDERS!r} + "/" + sc.name + "_" + "####"
bpy.ops.render.render(animation=True)
print("frames written to", {RENDERS!r})
"""
    run_script(script, timeout=3600)

## Gotchas worth knowing before you fight them

- **`/content` is ephemeral.** The VM dies after ~90 min idle / 12 h max. Anything you want to keep goes to Drive in the same cell that produces it.
- **Do not `pip install` Blender's own packages.** The `bpy` module carries its own bundled Python 3.11 stdlib in some builds; installing `numpy`/`scipy` into the notebook's interpreter can shadow Blender's and produce mismatched-ABI crashes. `bpy`'s wheel pins what it needs.
- **`import bpy` twice is fine, `bpy.ops` from a thread is not.** Anything touching `bpy.ops` must run on the main thread. That single rule is why the LLM‑Blender‑Agent's add‑on cannot simply be imported here — see `README.md`.
- **Blank renders** are almost always the camera (`scene.camera` unset, or aimed with wrong euler order) or a world strength of 0. Print `bpy.context.scene.camera` before blaming Cycles.
- **Out of memory on 4K**: lower `resolution_percentage`, not samples. Tile memory scales with pixels.